# Wordprint on Colab

用 Google 的 GPU 跑 Qwen 轉錄，不必在自己電腦安裝模型。音檔會上傳到這個 Colab 執行環境（Google 的機器）。

已預設開在 T4 GPU；若右上角不是 GPU，到「執行階段 → 變更執行階段類型」選 T4 GPU。

1. （要分說話者才需要）到 [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) 登入並接受使用條款，再於左側鑰匙圖示（Secrets）新增 `HF_TOKEN`（Hugging Face read token），並開啟「筆記本存取權」。沒有也能轉錄，只是沒有說話者。
2. 「執行階段 → 全部執行」，在第 3 格選音檔上傳。
3. 最後會下載 `normalized-run-*.json`；交給 Wordprint 繼續（驗證、閱讀頁、摘要、郵件）。

不在 Colab（本機 Jupyter，未實測）：第 1 格填 `AUDIO_FILES`，`HF_TOKEN` 設成環境變數，結果寫在目前資料夾。

In [ ]:
MODEL = "Qwen/Qwen3-ASR-1.7B"  # 或 "Qwen/Qwen3-ASR-0.6B"（較快、堪用）
MERGE = False  # 一次上傳多個檔案時，True＝依上傳順序合併成一段；False＝每個檔案各自一份結果
MAX_NEW_TOKENS = 2048  # 每段（≤3 分鐘）輸出上限；碰到上限的結果會標成 truncated
AUDIO_FILES = []  # 只在本機 Jupyter 用：音檔路徑（Colab 改用上傳）

In [ ]:
!pip install -q -U qwen-asr pyannote.audio

In [ ]:
import hashlib, json, subprocess
from pathlib import Path
import numpy as np
try:
    from google.colab import files
    uploaded = files.upload()  # {檔名: 原始位元組}；檔案同時存到目前目錄
except ImportError:  # 本機 Jupyter
    files, uploaded = None, {name: Path(name).read_bytes() for name in AUDIO_FILES}


def decode(name):
    """原始音檔 → 單聲道 16 kHz s16 PCM 位元組。"""
    return subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", name, "-ac", "1", "-ar", "16000", "-f", "s16le", "-"],
                          check=True, capture_output=True).stdout


groups = [list(uploaded)] if MERGE else [[name] for name in uploaded]
print("將轉錄：", groups)

In [ ]:
import importlib.metadata, unicodedata
import torch
from qwen_asr import Qwen3ASRModel

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32  # T4 沒有 bf16
ALIGNER = "Qwen/Qwen3-ForcedAligner-0.6B"
asr = Qwen3ASRModel.from_pretrained(MODEL, dtype=dtype, device_map=device, max_new_tokens=MAX_NEW_TOKENS,
                                    forced_aligner=ALIGNER, forced_aligner_kwargs={"dtype": dtype, "device_map": device})
generated = []  # transcribe() 不回傳停止原因；記下每批實際生成長度，碰到上限就標 truncated
_generate = asr.model.generate


def generate(*args, **kwargs):
    output = _generate(*args, **kwargs)
    generated.append(output.sequences.shape[1] - kwargs["input_ids"].shape[1])
    return output


asr.model.generate = generate


def gap(text):
    """對時器略過的空白、標點、符號原樣併入鄰詞（時間不變）；略過真的字則不併入、不借鄰詞時間，讓 validate 擋下。"""
    if any(not c.isspace() and unicodedata.category(c)[0] not in "PS" for c in text):  # transcript.py normalized，另略符號
        print(f"警告：對時漏掉「{text}」；這份結果驗證不會通過")
        return ""
    return text


def transcribe(names):
    """qwen-asr 自己把長音訊切成 ≤180 秒的段、逐段對時並換算成全段時間；這裡只把結果寫成 normalized run。"""
    pcm, sources = b"", []
    for name in names:
        data = decode(name)
        sources.append({"path": name, "sha256": hashlib.sha256(uploaded[name]).hexdigest(), "duration": len(data) / 32000,
                        "offset": len(pcm) / 32000, "sample_count": len(data) // 2, "sample_rate": 16000})
        pcm += data
    samples = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768
    generated.clear()
    result = asr.transcribe((samples, 16000), return_time_stamps=True)[0]
    words, cursor = [], 0
    for item in result.time_stamps or []:
        found = result.text.find(item.text, cursor)
        if found < 0:  # 對時文字與原文不符：保留原樣，讓 validate 如實擋下
            words.append({"text": item.text, "start": item.start_time, "end": item.end_time})
            continue
        words.append({"text": gap(result.text[cursor:found]) + item.text, "start": item.start_time, "end": item.end_time})
        cursor = found + len(item.text)
    if words:
        words[-1]["text"] += gap(result.text[cursor:])
    duration = len(samples) / 16000
    return {"schema_version": 1, "sources": sources, "platform": ("Google Colab " if files else "Jupyter ") + device,
            "model": {"id": MODEL, "size": MODEL.rsplit("-", 1)[1], "backend": "qwen-asr " + importlib.metadata.version("qwen-asr"),
                      "language": result.language, "dtype": str(dtype), "max_new_tokens": MAX_NEW_TOKENS},
            "alignment_model": {"id": ALIGNER},
            "chunks": [{"start": 0, "end": duration, "text": result.text, "words": words, "complete": True,
                        "truncated": max(generated, default=0) >= MAX_NEW_TOKENS, "generated_tokens": list(generated)}],
            "turns": [], "diarization_complete": False, "speakers": {}}, samples

In [ ]:
import os

DIARIZER = "pyannote/speaker-diarization-community-1"
try:
    from pyannote.audio import Pipeline
    if files:
        from google.colab import userdata
    token = userdata.get("HF_TOKEN") if files else os.environ["HF_TOKEN"]
    diarizer = Pipeline.from_pretrained(DIARIZER, token=token).to(torch.device(device))
except Exception as error:
    diarizer = None
    print(f"這次不分說話者（{type(error).__name__}）。要分說話者：到 https://huggingface.co/{DIARIZER} 登入並接受條款，"
          "在左側 Secrets 新增 HF_TOKEN 並開啟筆記本存取權（本機 Jupyter 改設環境變數 HF_TOKEN），再從這格往下重跑。")


def diarize(run, samples):
    if diarizer is None:
        return
    output = diarizer({"waveform": torch.from_numpy(samples)[None], "sample_rate": 16000})
    annotation = getattr(output, "speaker_diarization", output)
    order, duration = {}, len(samples) / 16000
    run["turns"] = [{"start": turn.start, "end": min(turn.end, duration), "speaker": order.setdefault(label, f"S{len(order) + 1}")}
                    for turn, _, label in annotation.itertracks(yield_label=True) if min(turn.end, duration) > turn.start]
    run["speakers"] = {speaker: {} for speaker in order.values()}
    run["diarization_complete"] = True
    run["diarization_model"] = {"id": DIARIZER, "backend": "pyannote.audio " + importlib.metadata.version("pyannote.audio")}

In [ ]:
for names in groups:
    run, samples = transcribe(names)
    diarize(run, samples)
    out = Path(f"normalized-run-{Path(names[0]).stem}.json")
    out.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(out, "：", len(run["chunks"][0]["words"]), "詞，說話者", len(run["speakers"]) or "未標記")
    if files:
        files.download(str(out))